## Feature Engineering
We will use inputs from `data_collection.ipynb` and output processed track and playlist features
so we can model playlist growth.

In [236]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

In [237]:
SNAPSHOT_DATE = max(f.replace("playlist_tracks_", "").replace(".csv","") for f in os.listdir("data/raw") if f.startswith("playlist_tracks_"))

raw_dir = "data/raw"

playlist_tracks_df = pd.read_csv(os.path.join(raw_dir, f"playlist_tracks_{SNAPSHOT_DATE}.csv"))
playlist_followers_df = pd.read_csv(os.path.join(raw_dir, f"playlist_followers_{SNAPSHOT_DATE}.csv"))
tracks_df = pd.read_csv(os.path.join(raw_dir, "tracks_features.csv"))
artists_df = pd.read_csv(os.path.join(raw_dir, "artists_features.csv"))

## uncomment
# playlist_tracks_df.head()
tracks_df.drop(columns=["artist_id", "album_id","track_id"], errors="ignore").head(), artists_df.drop(columns=["artist_id", "album_id","track_id"], errors="ignore").head()

(                        track_name  track_popularity  duration_ms  explicit  \
 0  Shattered (Turn The Car Around)                65       255493     False   
 1                   Stop And Stare                67       223853     False   
 2                   Don't Know Why                80       186251     False   
 3                        Breakeven                84       261426     False   
 4                        I'm Yours                84       242946     False   
 
                                           album_name release_date  \
 0                         All Sides (Deluxe Edition)   2008-07-15   
 1                                  Dreaming Out Loud   2007-01-01   
 2           Come Away With Me (Super Deluxe Edition)   2002-02-26   
 3                                         The Script   2008-07-14   
 4  We Sing. We Dance. We Steal Things. (Bonus Tra...   2008-05-01   
 
   release_date_precision  artist_name artist_name_artist  artist_popularity  \
 0              

### Data parsing & normalization

In [238]:
## How old is each track
playlist_tracks_df["added_at"] = pd.to_datetime(playlist_tracks_df["added_at"], errors="coerce")

def parse_release_date(row):
    try:
        if row["release_date_precision"] == "year":
            return datetime(int(row["release_date"]),1,1)
        elif row["release_date_precision"] == "month":
            return datetime.strptime(row["release_date"],"%Y-%m")
        else:
            return pd.to_datetime(row["release_date"])
    except Exception:
        return pd.NaT
tracks_df["release_date_parsed"] = tracks_df.apply(parse_release_date,axis=1)
tracks_df["track_age_days"] = (pd.to_datetime(SNAPSHOT_DATE) - tracks_df["release_date_parsed"]).dt.days

tracks_df[["track_name", "release_date", "track_age_days"]].head()

,track_name,release_date,track_age_days
0,Shattered (Turn The Car Around),2008-07-15,6474
1,Stop And Stare,2007-01-01,7035
2,Don't Know Why,2002-02-26,8805
3,Breakeven,2008-07-14,6475
4,I'm Yours,2008-05-01,6549


In [239]:
### Individual track features

In [240]:
track_features = tracks_df.copy()

In [241]:
track_features["log_artist_followers"] = np.log1p(track_features["artist_followers"])
track_features["explicit"] = track_features["explicit"].astype(int)
track_features.loc[track_features["track_age_days"] < 0, "track_age_days"] = np.nan
track_features_df = track_features[["track_id", 
                                    "track_popularity",
                                    "artist_popularity", 
                                    "log_artist_followers", 
                                    "track_age_days", 
                                    "explicit", 
                                    "genres"
                                ]]

track_features_df.drop(columns=["track_id", "explicit", "track_age_days"], errors="ignore").head()

,track_popularity,artist_popularity,log_artist_followers,genres
0,65,54,12.971359,jam band
1,67,82,16.776505,soft pop
2,80,73,15.055830,jazz pop
3,84,77,16.016669,NaN
4,84,73,15.813765,soft pop | acoustic pop


### Genre vectorization
Genres are converted into multi-hot columns.

In [242]:
track_features_df = track_features_df.copy()

In [243]:
track_features_df["genres_list"] = track_features_df["genres"].fillna("").str.split("|")

all_genres = sorted({ g for genres in track_features_df["genres_list"] for g in genres if g })
len(all_genres)

for g in all_genres:
    track_features_df[f"genre_{g}"] = track_features_df["genres_list"].apply(lambda lst: int(g in lst))

# drop raw genre
track_features_df = track_features_df.drop(columns=["genres","genres_list"])
track_features_df.drop(columns=["track_id"], errors="ignore").head()

,track_popularity,artist_popularity,log_artist_followers,track_age_days,explicit,genre_ acoustic pop,genre_ eurodance,genre_ eurodance,genre_ europop,genre_ folk,...,genre_pop punk,genre_post-grunge,genre_r&b,genre_rap,genre_rap,genre_reggae rock,genre_rock,genre_soft pop,genre_soft pop,genre_variété française
0,65,54,12.971359,6474.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,67,82,16.776505,7035.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,80,73,15.055830,8805.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,84,77,16.016669,6475.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,84,73,15.813765,6549.0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


### Merge tracks into playlist context
Each playlist is summarized as a distribution of its tracks.

In [244]:
playlist_track_features_df = playlist_tracks_df.merge(track_features_df, on="track_id", how="left")

In [ ]:
agg = { 
             "track_popularity": ["mean", "median", "std"],
             "artist_popularity": ["mean"], "log_artist_followers": ["mean"],
             "track_age_days": ["mean", "median"], "explicit": ["mean"],
        }

for g in all_genres:
    agg[f"genre_{g}"] = ["mean"]

playlist_features_df = (playlist_track_features_df.groupby("playlist_id").agg(agg))
playlist_features_df.columns = ["_".join(col).rstrip("_") for col in playlist_features_df.columns]
playlist_features_df = playlist_features_df.reset_index()

In [246]:
# artist diversity
playlist_size = (playlist_tracks_df.groupby("playlist_id")["track_id"].count().rename("playlist_size"))
artist_diversity = (playlist_tracks_df.groupby("playlist_id")["artist_id"].nunique().rename("num_unique_artists"))

playlist_features_df = playlist_features_df.merge(
    playlist_size, on="playlist_id", how="left").merge(
    artist_diversity, on="playlist_id", how="left"
)
playlist_features_df["artist_diversity_ratio"] = (playlist_features_df["num_unique_artists"] / playlist_features_df["playlist_size"])

# followers
playlist_features_df = playlist_features_df.merge(
    playlist_followers_df[["playlist_id", "followers"]],
    on="playlist_id", how="left"
)
playlist_features_df.drop(columns=["playlist_id"], errors="ignore").head()

,track_popularity_mean,track_popularity_median,track_popularity_std,artist_popularity_mean,log_artist_followers_mean,track_age_days_mean,track_age_days_median,explicit_mean,genre_ acoustic pop_mean,genre_ eurodance_mean,...,genre_rap _mean,genre_reggae rock _mean,genre_rock_mean,genre_soft pop_mean,genre_soft pop _mean,genre_variété française_mean,playlist_size,num_unique_artists,artist_diversity_ratio,followers
0,72.217687,75.0,13.069109,72.544218,15.340625,5999.741497,5851.0,0.054422,0.006803,0.006803,...,0.006803,0.013605,0.006803,0.061224,0.006803,0.006803,147,101,0.687075,27


### Save feature tables

In [247]:
out_dir = "data/processed"

os.makedirs(out_dir, exist_ok=True)
track_features_path = os.path.join(out_dir, "track_features_processed.csv")
playlist_features_path = os.path.join(out_dir, "playlist_features.csv")

track_features_df.to_csv(track_features_path, index=False)
playlist_features_df.to_csv(playlist_features_path, index=False)
track_features_path, playlist_features_path

print("Track features:", track_features_df.shape)
print("Playlist features:", playlist_features_df.shape)

## check for null playlist feature values
# playlist_features_df.isna().mean().sort_values(ascending=False).head(10)

Track features: (141, 45)
Playlist features: (1, 52)
